<a href="https://colab.research.google.com/github/jmmontestolentino-cyber/nfl-telemetry-dlt-pipeline/blob/main/src/00_data_simulator/nfl_telemetry_generator.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
pip install confluent-kafka

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.8/4.8 MB 17.7 MB/s eta 0:00:00


In [ ]:
key=""
secret=""

In [ ]:
import json
import random
import time
from confluent_kafka import Producer

# ==============================================================================
# 1. PARÁMETROS DE PRUEBA (AJUSTA ESTO PARA TUS SIMULACIONES)
# ==============================================================================
# Define cuántos minutos exactos quieres que corra el script. Acepta decimales.
MINUTOS_DE_SIMULACION = 20

# ==============================================================================
# 2. CONFIGURACIÓN DE CONEXIÓN A CONFLUENT CLOUD
# ==============================================================================
conf = {
    'bootstrap.servers': '',
    'security.protocol': '',
    'sasl.mechanisms': '',
    'sasl.username': key,
    'sasl.password': secret
}

producer = Producer(conf)

def delivery_callback(err, msg):
    if err:
        print(f'Error al enviar: {err}')

# ==============================================================================
# 3. BASE DE DATOS DE 22 JUGADORES (KC OFFENSE vs BAL DEFENSE)
# Línea de golpeo aproximada en la yarda 50
# ==============================================================================
players_pool = [
    # --- OFENSIVA DE LOS CHIEFS (KC) ---
    {"id": "15_Mahomes", "name": "Patrick Mahomes", "pos": "QB", "team": "KC", "x": 45.0, "y": 26.6},
    {"id": "10_Pacheco", "name": "Isiah Pacheco", "pos": "RB", "team": "KC", "x": 43.5, "y": 26.6},
    {"id": "87_Kelce", "name": "Travis Kelce", "pos": "TE", "team": "KC", "x": 49.0, "y": 20.0},
    {"id": "04_Rice", "name": "Rashee Rice", "pos": "WR", "team": "KC", "x": 49.0, "y": 42.0},
    {"id": "01_Worthy", "name": "Xavier Worthy", "pos": "WR", "team": "KC", "x": 49.0, "y": 10.0},
    {"id": "52_Humphrey", "name": "Creed Humphrey", "pos": "C", "team": "KC", "x": 49.5, "y": 26.6},
    {"id": "65_Smith", "name": "Trey Smith", "pos": "RG", "team": "KC", "x": 49.5, "y": 28.0},
    {"id": "74_Taylor", "name": "Jawaan Taylor", "pos": "RT", "team": "KC", "x": 49.5, "y": 30.0},
    {"id": "62_Thuney", "name": "Joe Thuney", "pos": "LG", "team": "KC", "x": 49.5, "y": 25.0},
    {"id": "76_Suamataia", "name": "Kingsley Suamataia", "pos": "LT", "team": "KC", "x": 49.5, "y": 23.0},
    {"id": "83_Gray", "name": "Noah Gray", "pos": "TE", "team": "KC", "x": 49.0, "y": 33.0},

    # --- DEFENSIVA DE LOS RAVENS (BAL) ---
    {"id": "92_Madubuike", "name": "Justin Madubuike", "pos": "DT", "team": "BAL", "x": 51.0, "y": 25.0},
    {"id": "98_Jones", "name": "Travis Jones", "pos": "DT", "team": "BAL", "x": 51.0, "y": 28.0},
    {"id": "53_VanNoy", "name": "Kyle Van Noy", "pos": "OLB", "team": "BAL", "x": 51.5, "y": 20.0},
    {"id": "99_Oweh", "name": "Odafe Oweh", "pos": "OLB", "team": "BAL", "x": 51.5, "y": 33.0},
    {"id": "00_Smith", "name": "Roquan Smith", "pos": "ILB", "team": "BAL", "x": 55.0, "y": 26.6},
    {"id": "40_Harrison", "name": "Malik Harrison", "pos": "ILB", "team": "BAL", "x": 55.0, "y": 23.0},
    {"id": "14_Hamilton", "name": "Kyle Hamilton", "pos": "SS", "team": "BAL", "x": 58.0, "y": 30.0},
    {"id": "32_Williams", "name": "Marcus Williams", "pos": "FS", "team": "BAL", "x": 65.0, "y": 26.6},
    {"id": "16_Humphrey", "name": "Marlon Humphrey", "pos": "CB", "team": "BAL", "x": 52.0, "y": 10.0},
    {"id": "21_Stephens", "name": "Brandon Stephens", "pos": "CB", "team": "BAL", "x": 52.0, "y": 42.0},
    {"id": "39_Washington", "name": "Ar'Darius Washington", "pos": "DB", "team": "BAL", "x": 56.0, "y": 15.0}
]

topic = 'nfl_telemetry'
game_id = "2026_01_KC_BAL"

# ==============================================================================
# 4. MOTOR DEL PARTIDO CONTINUO
# ==============================================================================
print(f"🏈 Iniciando simulación de {MINUTOS_DE_SIMULACION} minutos (Presiona Ctrl+C para detener antes)...")

numero_jugada_actual = 101
estado_partido = "HUDDLE"
tiempo_proximo_cambio = time.time() + 5
tiempo_fin_simulacion = time.time() + (MINUTOS_DE_SIMULACION * 60)

try:
    # El bucle ahora se detiene cuando alcanzamos el tiempo límite
    while time.time() < tiempo_fin_simulacion:
        current_time_ms = int(time.time() * 1000)

        # 1. Lógica de cambio de estado (Jugada vs Descanso)
        if time.time() >= tiempo_proximo_cambio:
            if estado_partido == "HUDDLE":
                estado_partido = "ACTIVO"
                duracion_fase = random.randint(5, 12)
                print(f"\n🟢 ¡SNAP! Inicia play_{numero_jugada_actual} (Duración: {duracion_fase}s)")
            else:
                estado_partido = "HUDDLE"
                duracion_fase = random.randint(25, 40)
                print(f"🛑 Fin de la jugada. Equipos en Huddle (Duración: {duracion_fase}s)")
                numero_jugada_actual += 1

            tiempo_proximo_cambio = time.time() + duracion_fase

        # 2. Generación de datos condicionada por el estado
        if estado_partido == "ACTIVO":
            current_play_id = f"play_{numero_jugada_actual}"
            rango_velocidad = (5.0, 21.5)
            rango_aceleracion = (-4.0, 5.0)
            movimiento = 1.5
        else:
            current_play_id = f"huddle_pre_{numero_jugada_actual}"
            rango_velocidad = (0.0, 3.5)
            rango_aceleracion = (-1.0, 1.0)
            movimiento = 0.2

        # 3. Envío continuo de telemetría de los 22 jugadores
        for player in players_pool:
            player["x"] = max(0.0, min(120.0, player["x"] + random.uniform(-movimiento, movimiento)))
            player["y"] = max(0.0, min(53.3, player["y"] + random.uniform(-movimiento, movimiento)))

            telemetry_data = {
                "game_id": game_id,
                "play_id": current_play_id,
                "play_status": estado_partido,
                "player_id": player["id"],
                "player_name": player["name"],
                "position": player["pos"],
                "team": player["team"],
                "x_coord": round(player["x"], 2),
                "y_coord": round(player["y"], 2),
                "speed_mph": round(random.uniform(*rango_velocidad), 1),
                "acceleration_m_s2": round(random.uniform(*rango_aceleracion), 2),
                "heart_rate_bpm": random.randint(130 if estado_partido == "HUDDLE" else 150, 185),
                "stamina_pct": round(random.uniform(65.0, 98.0), 1),
                "timestamp": current_time_ms
            }

            producer.produce(topic, key=player["id"], value=json.dumps(telemetry_data), callback=delivery_callback)

        producer.poll(0)

        # Transmisión ininterrumpida a 5Hz
        time.sleep(0.2)

    print(f"\n✅ Tiempo de simulación ({MINUTOS_DE_SIMULACION} min) completado exitosamente.")

except KeyboardInterrupt:
    print("\n⚠️ Apagando receptores de sensores de forma manual...")
finally:
    producer.flush()
    print("Simulador cerrado.")

🏈 Iniciando simulación de 20 minutos (Presiona Ctrl+C para detener antes)...

🟢 ¡SNAP! Inicia play_101 (Duración: 9s)
🛑 Fin de la jugada. Equipos en Huddle (Duración: 32s)

🟢 ¡SNAP! Inicia play_102 (Duración: 6s)
🛑 Fin de la jugada. Equipos en Huddle (Duración: 30s)

🟢 ¡SNAP! Inicia play_103 (Duración: 5s)
🛑 Fin de la jugada. Equipos en Huddle (Duración: 32s)

🟢 ¡SNAP! Inicia play_104 (Duración: 12s)
🛑 Fin de la jugada. Equipos en Huddle (Duración: 33s)

🟢 ¡SNAP! Inicia play_105 (Duración: 11s)
🛑 Fin de la jugada. Equipos en Huddle (Duración: 27s)

🟢 ¡SNAP! Inicia play_106 (Duración: 9s)
🛑 Fin de la jugada. Equipos en Huddle (Duración: 39s)

🟢 ¡SNAP! Inicia play_107 (Duración: 11s)
🛑 Fin de la jugada. Equipos en Huddle (Duración: 40s)

🟢 ¡SNAP! Inicia play_108 (Duración: 9s)
🛑 Fin de la jugada. Equipos en Huddle (Duración: 37s)

🟢 ¡SNAP! Inicia play_109 (Duración: 5s)
🛑 Fin de la jugada. Equipos en Huddle (Duración: 29s)

🟢 ¡SNAP! Inicia play_110 (Duración: 10s)
🛑 Fin de la jugada. Equ